# Lab 01 · Better prompts and small evaluations

**Prepared by Mustafa Assaf** · 5–10 minute example

## Setup
Run this cell once. It loads the reusable helpers; the example follows below. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Make the criteria explicit, then test them

Compare a basic prompt with one that adds our rule, a worked example, and explicit criteria. Both request JSON containing a decision and a brief reason.

This uses decomposition into two checkable criteria, often associated with chain-of-thought prompting. Request a short explanation, not a long private reasoning trace. ReAct adds actions and observations in Lab 04, after tools.

The three cases include the boundary at team size 8 and a case where only one criterion holds.

In [ ]:
import json
rule = 'Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith.'
cases = [(3, False, "modular monolith"), (8, True, "microservices"), (8, False, "modular monolith")]
scores = {"basic": 0, "guided": 0}
for size, independent, expected in cases:
    case = {"team_size": size, "independent_deploys": independent}
    basic = f"Choose modular monolith or microservices for {case}. Return JSON with decision and a brief reason."
    guided = rule + " Example: team_size=2, independent_deploys=True -> modular monolith. Check both criteria. " + basic
    for name, prompt in {"basic": basic, "guided": guided}.items():
        fixture = {"decision": expected if name == "guided" else "modular monolith", "reason": "Both criteria must hold."}
        text = ask(prompt, example=json.dumps(fixture))
        print(name, case, text)
        try: result = json.loads(text)
        except json.JSONDecodeError: result = {}
        scores[name] += int(isinstance(result, dict) and result.get("decision") == expected)
print("Policy matches:", scores, "out of", len(cases))
assert scores["guided"] == len(cases), "Inspect the failed policy match."

**Expected practice output:** `basic: 2`, `guided: 3`. The basic fixture intentionally gives the wrong answer once, so the evaluator detects it. These scripted scores are not model results or evidence that a prompt is better.

In live mode, compare policy matches and manually check each brief reason. Three cases measure limited policy adherence, not real architecture quality. Malformed JSON scores zero; a wrong decision fails the final check.

**Small exercise:** add `(7, True, "modular monolith")` and try the comparison live. Does either prompt violate the threshold?

## Read further

- [Chain-of-thought prompting](https://arxiv.org/abs/2201.11903)
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering)